In [0]:
hr_bronze = spark.table(
    "health_and_habit_guardian.bronze.heart_rate"
)

In [0]:
from pyspark.sql import functions as F

# store all timestamps in UTC; fix the Spark session time zone so that
# _processed_at and any timestamp casts are written consistently in UTC
spark.conf.set("spark.sql.session.timeZone", "UTC")

# explicitly cast types in the Silver layer to estabilish a canonical schema

hr = (
    hr_bronze
    .withColumn(
        "heart_rate_bpm",
        F.col("heart_rate_bpm").cast("int")
    )
)

In [0]:
from pyspark.sql.window import Window

# heart-rate deduplication

window = Window.partitionBy(
    "user_id",
    "timestamp",
    "source"
).orderBy(
    F.col("_ingested_at").desc()
)

In [0]:
# adding a quality status

hr = (
    hr
    .withColumn(
        "_quality_status",
        F.when(
            F.col("user_id").isNull() |
            F.col("timestamp").isNull() |
            (F.col("heart_rate_bpm") < 30) |
            (F.col("heart_rate_bpm") > 250),
            F.lit("INVALID")
        ).otherwise(F.lit("VALID"))
    )
)

In [0]:
hr = hr.withColumn(
    "_processed_at",
    F.current_timestamp()
)

In [0]:
# validate activity_states

valid_states = [
    "resting",
    "walking",
    "running",
    "cycling",
    "sleeping",
    "unknown"
]

hr = hr.withColumn(
    "_invalid_activity_state",
    ~F.col("activity_state").isin(valid_states)
)

# fold the activity_state check into the quality status
# so unexpected states are quarantined instead of reaching Silver
hr = hr.withColumn(
    "_quality_status",
    F.when(
        F.col("_invalid_activity_state"),
        F.lit("INVALID")
    ).otherwise(F.col("_quality_status"))
)

In [0]:
# upsert the deduplicated + valid records to Silver;
# quarantine still sees every invalid record (including duplicates)

hr_deduped = (
    hr
    .withColumn(
        "_row_number",
        F.row_number().over(window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

valid_hr = (
    hr_deduped
    .filter(F.col("_quality_status") == "VALID")
)

invalid_hr = (
    hr
    .filter(F.col("_quality_status") == "INVALID")
)

In [0]:
(
    invalid_hr
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "health_and_habit_guardian.silver.quarantine_heart_rate"
    )
)

In [0]:
# idempotent upsert on the business key (user_id, timestamp, source)
# so late-arriving data updates the existing Silver record

valid_hr.createOrReplaceTempView("hr_updates")

spark.sql("""
CREATE TABLE IF NOT EXISTS health_and_habit_guardian.silver.heart_rate
USING DELTA
AS SELECT * FROM hr_updates WHERE 1 = 0
""")

# dedupe the source inside the MERGE so the upsert can never fail with
# [DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE]; latest ingestion wins
cols = ", ".join(valid_hr.columns)

spark.sql(f"""
MERGE INTO health_and_habit_guardian.silver.heart_rate AS target
USING (
    SELECT {cols}
    FROM (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY user_id, timestamp, source
            ORDER BY _ingested_at DESC NULLS LAST
        ) AS _rn
        FROM hr_updates
    )
    WHERE _rn = 1
) AS source
ON target.user_id = source.user_id
   AND target.timestamp = source.timestamp
   AND target.source = source.source
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")


In [0]:
%run ./00_quality_utils


In [0]:
write_quality_summary(
    df=hr,
    table_name="heart_rate",
    date_col="timestamp",
    dedup_cols=["user_id", "timestamp", "source"],
)
